<a href="https://colab.research.google.com/github/Santhosh-S-6103/Blood-Group-Detection/blob/main/Blood_Grp_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
import tensorflow as tf
import tensorflow.keras as k
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, AveragePooling2D, Flatten, Dropout

In [ ]:
path_folder = "/content/drive/MyDrive/bldgrp/dataset_blood_group"
class_name = os.listdir(path_folder)
class_name

In [ ]:
path_folder = "/content/drive/MyDrive/bldgrp/dataset_blood_group"
class_name = os.listdir(path_folder)
class_name.sort()
print(class_name)
image_data = []
label_data = []
count = 0
for folder in class_name:
    images = os.listdir(path_folder + "/" + folder)
    print("Loading Folder -- {} " .format(folder), "The Count of Classes ==> ",count)
    for img in images:
        image = cv2.imread(path_folder + "/" + folder + "/" + img)
        image = cv2.resize(image, (224, 224))

        image_data.append(image)
        label_data.append(count)
    count += 1
print("---- Done ----------- ")

In [ ]:
data = np.array(image_data)
data = data.astype("float32")
data = data/255.0

label = np.array(label_data)

In [ ]:
print(data.shape)

In [ ]:
label_num = to_categorical(label, len(class_name))

In [ ]:
x_img, y_img = shuffle(data, label_num)
x_train, x_test, y_train, y_test = train_test_split(x_img, y_img, train_size=0.8)

In [ ]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(0, 25):
    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(x_train[i])
    plt.title(class_name[np.argmax(y_train[i])])

In [ ]:
model = k.models.Sequential()

model.add(k.layers.Conv2D(16, (5, 5), activation="relu", input_shape=(224, 224, 3), padding="same"))
model.add(k.layers.AveragePooling2D((2, 2)))

model.add(k.layers.Conv2D(32, (4, 4), activation="relu",  padding="same"))
# model.add(k.layers.BatchNormalization())
model.add(k.layers.AveragePooling2D((2, 2)))


model.add(k.layers.Conv2D(64, (3, 3), activation="relu", padding="same"))
model.add(k.layers.AveragePooling2D((2, 2)))

model.add(k.layers.Conv2D(128, (2, 2), activation="relu", padding="same"))
model.add(k.layers.MaxPool2D((2, 2)))

model.add(k.layers.Flatten())

model.add(k.layers.Dense(256, activation="relu"))
# model.add(k.layers.BatchNormalization())
model.add(k.layers.Dropout(0.5))

model.add(k.layers.Dense(32, activation="relu"))
model.add(k.layers.Dropout(0.2))

model.add(k.layers.Dense(len(class_name), activation="softmax"))

model.compile(optimizer="adam", loss=k.losses.CategoricalCrossentropy(), metrics=["accuracy"])

model.summary()

In [ ]:
history = model.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test), validation_split=0.2)

In [ ]:
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.show()
plt.savefig('LossVal_loss')

In [ ]:
# accuracies
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.show()
plt.savefig('AccVal_acc')

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Assuming 'model' is your trained model
y_pred_probabilities = model.predict(x_train)
y_pred = np.argmax(y_pred_probabilities, axis=1)  # Convert probabilities to class labels

# Assuming y_test is one-hot encoded
y_true = np.argmax(y_train, axis=1)  # Convert one-hot encoded labels to class labels

# Print classification report
print(classification_report(y_true, y_pred))



In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Assuming 'model' is your trained model
y_pred_probabilities = model.predict(x_test)
y_pred = np.argmax(y_pred_probabilities, axis=1)  # Convert probabilities to class labels

# Assuming y_test is one-hot encoded
y_true = np.argmax(y_test, axis=1)  # Convert one-hot encoded labels to class labels

# Print classification report
print(classification_report(y_true, y_pred))

In [29]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.datasets import ImageFolder
import torch.optim as optim
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
from PIL import Image

# Custom dataset class
class CustomImageFolder(Dataset):
    def __init__(self, root_dir, transform):
        self.dataset = ImageFolder(root_dir, transform=transform)
        self.corrupted_idx = []

    def __getitem__(self, index):
        try:
            return self.dataset[index]
        except OSError:
            self.corrupted_idx.append(index)
            return None  # Return None for corrupted file

    def __len__(self):
        return len(self.dataset)

    def get_corrupted_files(self):
        return [self.dataset.imgs[i] for i in self.corrupted_idx]

# Custom collate function to filter out None values
def custom_collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    return torch.utils.data.dataloader.default_collate(batch)

# Data preprocessing
transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Custom dataset
custom_dataset = CustomImageFolder('/content/drive/MyDrive/bldgrp/dataset_blood_group', transform=transform)
train_size = int(0.8 * len(custom_dataset))
test_size = len(custom_dataset) - train_size
train_dataset, test_dataset = random_split(custom_dataset, [train_size, test_size])

# DataLoaders with custom collate function
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)

# Model setup
alexnet = models.alexnet(pretrained=True)
num_classes = len(custom_dataset.dataset.classes)
alexnet.classifier[6] = nn.Linear(alexnet.classifier[6].in_features, num_classes)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(alexnet.parameters(), lr=0.001, momentum=0.9)
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alexnet.to(device)

# Training the model
num_epochs = 15
for epoch in range(num_epochs):
    alexnet.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = alexnet(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

    # Testing the model
    alexnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = alexnet(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Accuracy of the network on the test images: {100 * correct / total} %')

print('Finished Training')

# Print corrupted files
corrupted_files = custom_dataset.get_corrupted_files()
print("Corrupted files:", corrupted_files)


Epoch 1, Loss: 0.9153682299672741
Accuracy of the network on the test images: 85.39518900343643 %
Epoch 2, Loss: 0.3842443968743494
Accuracy of the network on the test images: 90.54982817869416 %
Epoch 3, Loss: 0.31054236494923293
Accuracy of the network on the test images: 90.72164948453609 %
Epoch 4, Loss: 0.2812303259764632
Accuracy of the network on the test images: 92.43986254295532 %


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

train_losses = []
train_accuracies = []
test_accuracies = []

# Train the model
for epoch in range(num_epochs):
    alexnet.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = alexnet(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct_train / total_train
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    # Test the model
    alexnet.eval()
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = alexnet(images)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_accuracy = 100 * correct_test / total_test
    test_accuracies.append(test_accuracy)


# Plotting Loss and Accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(test_accuracies, label='Testing Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Testing Accuracy Over Epochs')
plt.legend()

plt.show()

In [ ]:
torch.save(alexnet.state_dict(), '/content/drive/MyDrive/BG.pth')

torch.save(alexnet, 'BG.pth')

In [ ]:
alexnet = models.alexnet()
alexnet.classifier[6] = nn.Linear(alexnet.classifier[6].in_features, len(class_name))
alexnet.load_state_dict(torch.load('/content/drive/MyDrive/BG.pth'))
alexnet.to(device)  # Move the model to the device

In [ ]:
from PIL import Image
import torch
import torchvision.transforms as transforms

# Define your transformation pipeline
transform = transforms.Compose([
    transforms.Resize((227, 227)),  # AlexNet uses 227x227 input size
    transforms.ToTensor(),
    # Normalization values for pretrained models are usually the means and stds of the ImageNet dataset.
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict_single_image(image_path, model, transform, device):
    # Open the image file
    image = Image.open(image_path)

    # Convert to RGB if not already (assumes that the input image can have 1 or 3 channels)
    if image.mode != 'RGB':
        image = image.convert('RGB')

    # Apply the transformations to the image
    image = transform(image)

    # Add a batch dimension since pytorch expects a batch, not a single image
    image = image.unsqueeze(0).to(device)

    # Set the model to evaluation mode
    model.eval()

    # No need to track gradients for validation, hence wrap in torch.no_grad()
    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)

    # Get the index of the predicted class
    return predicted.item()

# Make sure to define `alexnet`, `device`, and `dataset.classes` appropriately before calling this function
# Example usage:
image_path = '/content/drive/MyDrive/bldgrp/dataset_blood_group/B-/cluster_3_1035.BMP'  # Fixed path by adding '/bldgrp'
predicted_class_index = predict_single_image(image_path, alexnet, transform, device)

# Get the list of classes from our custom_dataset
classes = custom_dataset.dataset.classes
predicted_class = classes[predicted_class_index]
print(f'Predicted Class: {predicted_class}')